In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv(r"C:\Users\VISHNU\Downloads\Datheon_internship\Day-1- SuperStore_Sales\Data\superstore_sales.csv")

In [4]:
df.shape
df.head()
df.columns.to_list()
df.isna().sum(),df.dtypes

(Row ID             0
 Order ID           0
 Order Date       806
 Ship Date        806
 Ship Mode        806
 Customer ID      806
 Customer Name    806
 Segment          806
 Country          806
 City             806
 State            806
 Postal Code      817
 Region           806
 Product ID       806
 Category         806
 Sub-Category     806
 Product Name     806
 Sales            806
 Quantity         806
 Discount         806
 Profit           806
 dtype: int64,
 Row ID               str
 Order ID             str
 Order Date           str
 Ship Date            str
 Ship Mode            str
 Customer ID          str
 Customer Name        str
 Segment              str
 Country              str
 City                 str
 State                str
 Postal Code      float64
 Region               str
 Product ID           str
 Category             str
 Sub-Category         str
 Product Name         str
 Sales            float64
 Quantity         float64
 Discount         float64
 Pr

In [9]:
#characterize missing values
df.tail(), df.isna().sum().sort_values(ascending=False).head(10), df['Order Date'].dropna().head()


(      Row ID        Order ID Order Date Ship Date Ship Mode Customer ID  \
 10795    Yes  US-2018-147886        NaN       NaN       NaN         NaN   
 10796    Yes  US-2018-147998        NaN       NaN       NaN         NaN   
 10797    Yes  US-2018-151127        NaN       NaN       NaN         NaN   
 10798    Yes  US-2018-155999        NaN       NaN       NaN         NaN   
 10799    Yes  US-2018-155999        NaN       NaN       NaN         NaN   
 
       Customer Name Segment Country City  ... Postal Code  Region Product ID  \
 10795           NaN     NaN     NaN  NaN  ...         NaN     NaN        NaN   
 10796           NaN     NaN     NaN  NaN  ...         NaN     NaN        NaN   
 10797           NaN     NaN     NaN  NaN  ...         NaN     NaN        NaN   
 10798           NaN     NaN     NaN  NaN  ...         NaN     NaN        NaN   
 10799           NaN     NaN     NaN  NaN  ...         NaN     NaN        NaN   
 
       Category Sub-Category Product Name Sales  Quant

In [8]:
df[df['Order Date'].isna()].head(10), df[df['Order Date'].isna()]['Row ID'].value_counts(dropna=False), df['Row ID'].tail(15).tolist()


(                  Row ID        Order ID Order Date Ship Date Ship Mode  \
 9994              Person          Region        NaN       NaN       NaN   
 9995       Anna Andreadi            West        NaN       NaN       NaN   
 9996         Chuck Magee            East        NaN       NaN       NaN   
 9997      Kelly Williams         Central        NaN       NaN       NaN   
 9998   Cassandra Brandow           South        NaN       NaN       NaN   
 9999            Returned        Order ID        NaN       NaN       NaN   
 10000                Yes  CA-2015-100762        NaN       NaN       NaN   
 10001                Yes  CA-2015-100762        NaN       NaN       NaN   
 10002                Yes  CA-2015-100762        NaN       NaN       NaN   
 10003                Yes  CA-2015-100762        NaN       NaN       NaN   
 
       Customer ID Customer Name Segment Country City  ... Postal Code  Region  \
 9994          NaN           NaN     NaN     NaN  NaN  ...         NaN     NaN  

In [10]:
clean=df.copy()
clean['Order Date']=pd.to_datetime(clean['Order Date'], errors='coerce')
clean['Ship Date']=pd.to_datetime(clean['Ship Date'], errors='coerce')
valid=clean[clean['Order Date'].notna() & clean['Sales'].notna() & clean['Profit'].notna()].copy()
valid.shape, valid[['Order Date','Sales','Profit','Discount','Quantity']].describe(), valid['Order Date'].min(), valid['Order Date'].max()


((9994, 21),
                        Order Date         Sales       Profit     Discount  \
 count                        9994   9994.000000  9994.000000  9994.000000   
 mean   2017-04-30 05:17:08.056834    229.858001    28.656896     0.156203   
 min           2015-01-03 00:00:00      0.444000 -6599.978000     0.000000   
 25%           2016-05-23 00:00:00     17.280000     1.728750     0.000000   
 50%           2017-06-26 00:00:00     54.490000     8.666500     0.200000   
 75%           2018-05-14 00:00:00    209.940000    29.364000     0.200000   
 max           2018-12-30 00:00:00  22638.480000  8399.976000     0.800000   
 std                           NaN    623.245101   234.260108     0.206452   
 
           Quantity  
 count  9994.000000  
 mean      3.789574  
 min       1.000000  
 25%       2.000000  
 50%       3.000000  
 75%       5.000000  
 max      14.000000  
 std       2.225110  ,
 Timestamp('2015-01-03 00:00:00'),
 Timestamp('2018-12-30 00:00:00'))

In [11]:
valid['YearMonth']=valid['Order Date'].dt.to_period('M').astype(str)
monthly=valid.groupby('YearMonth').agg(Sales=('Sales','sum'),Profit=('Profit','sum'),Orders=('Order ID','nunique')).reset_index()
cat=valid.groupby('Category').agg(Sales=('Sales','sum'),Profit=('Profit','sum'),Quantity=('Quantity','sum')).sort_values('Sales',ascending=False)
reg=valid.groupby('Region').agg(Sales=('Sales','sum'),Profit=('Profit','sum'))
seg=valid.groupby('Segment').agg(Sales=('Sales','sum'),Profit=('Profit','sum'),Customers=('Customer ID','nunique'))
disc=valid.groupby('Discount').agg(Transactions=('Discount','size'),Sales=('Sales','sum'),Profit=('Profit','sum'),Avg_Profit=('Profit','mean')).reset_index()
total_sales=valid.Sales.sum(); total_profit=valid.Profit.sum()
total_sales,total_profit,valid['Order ID'].nunique(),valid['Customer ID'].nunique(),cat,reg,seg,disc, monthly.sort_values('Sales',ascending=False).head()


(np.float64(2297200.8603),
 np.float64(286397.0217),
 5009,
 793,
                        Sales       Profit  Quantity
 Category                                           
 Technology       836154.0330  145454.9481    6939.0
 Furniture        741999.7953   18451.2728    8028.0
 Office Supplies  719047.0320  122490.8008   22906.0,
                Sales       Profit
 Region                           
 Central  501239.8908   39706.3625
 East     678781.2400   91522.7800
 South    391721.9050   46749.4303
 West     725457.8245  108418.4489,
                     Sales       Profit  Customers
 Segment                                          
 Consumer     1.161401e+06  134119.2092        409
 Corporate    7.061464e+05   91979.1340        236
 Home Office  4.296531e+05   60298.6785        148,
     Discount  Transactions         Sales       Profit  Avg_Profit
 0       0.00          4798  1.087908e+06  320987.6032   66.900292
 1       0.10            94  5.436935e+04    9029.1770   96.055074


In [ ]:
import nbformat, os, textwrap, json
from pathlib import Path

out = Path.cwd()
out.mkdir(parents=True, exist_ok=True)
nb = nbformat.v4.new_notebook()
nb.metadata = {
    "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
    "language_info": {"name": "python", "version": "3"}
}
cells = []

cells.append(nbformat.v4.new_markdown_cell("""# Exploratory Data Analysis (EDA) – Superstore Retail Sales

**Objective:** Analyze customer purchase volume, monthly seasonality, category-wise revenue, regional/segment performance, and discount sensitivity.

**Dataset:** `superstore_sales.csv`  
**Libraries:** Pandas, NumPy, Matplotlib, Seaborn

> Run the notebook from top to bottom. Keep `superstore_sales.csv` in the same folder as this notebook."""))

cells.append(nbformat.v4.new_code_cell("""# 1. Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Load the dataset
file_path = Path("superstore_sales.csv")
df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
display(df.head())"""))

cells.append(nbformat.v4.new_markdown_cell("## 2. Initial data inspection"))
cells.append(nbformat.v4.new_code_cell("""print("Shape:", df.shape)
print("\\\\nColumn names:")
print(df.columns.tolist())

print("\\\\nData types:")
display(df.dtypes.to_frame("Data Type"))

print("\\\\nMissing values:")
missing = df.isna().sum().to_frame("Missing Values")
missing["Missing %"] = (missing["Missing Values"] / len(df) * 100).round(2)
display(missing.sort_values("Missing Values", ascending=False))

print("\\\\nExact duplicate rows:", df.duplicated().sum())
display(df.describe(include="all").T)"""))

cells.append(nbformat.v4.new_markdown_cell("""## 3. Data cleaning and validation

The supplied file contains 10,800 rows, but only 9,994 rows have valid order dates and numeric sales/profit values. The remaining 806 rows are incomplete/non-transaction records (including return-related rows). They are excluded from sales analysis.

The code below parses dates, converts numeric fields, removes exact duplicates, and retains valid sales transactions."""))
cells.append(nbformat.v4.new_code_cell("""# Work on a copy
data = df.copy()

# Parse dates and numeric columns
data["Order Date"] = pd.to_datetime(data["Order Date"], errors="coerce")
data["Ship Date"] = pd.to_datetime(data["Ship Date"], errors="coerce")

numeric_cols = ["Row ID", "Postal Code", "Sales", "Quantity", "Discount", "Profit"]
for col in numeric_cols:
    data[col] = pd.to_numeric(data[col], errors="coerce")

# Remove exact duplicate records
data = data.drop_duplicates()

# Keep complete transaction rows needed for this analysis
required_cols = ["Order ID", "Order Date", "Customer ID", "Category",
                 "Sales", "Quantity", "Discount", "Profit"]
sales = data.dropna(subset=required_cols).copy()

# Additional date features
sales["Year"] = sales["Order Date"].dt.year
sales["Month"] = sales["Order Date"].dt.month
sales["Month Name"] = sales["Order Date"].dt.strftime("%b")
sales["Year-Month"] = sales["Order Date"].dt.to_period("M").astype(str)

print("Rows before cleaning:", len(df))
print("Rows after cleaning:", len(sales))
print("Rows excluded:", len(df) - len(sales))
print("Cleaned dataset shape:", sales.shape)
display(sales.head())"""))

cells.append(nbformat.v4.new_markdown_cell("## 4. Key performance indicators (KPIs)"))
cells.append(nbformat.v4.new_code_cell("""total_sales = sales["Sales"].sum()
total_profit = sales["Profit"].sum()
total_orders = sales["Order ID"].nunique()
total_customers = sales["Customer ID"].nunique()
total_quantity = sales["Quantity"].sum()
profit_margin = total_profit / total_sales * 100

kpis = pd.DataFrame({
    "Metric": ["Total Sales", "Total Profit", "Unique Orders",
               "Unique Customers", "Units Sold", "Profit Margin (%)"],
    "Value": [total_sales, total_profit, total_orders,
              total_customers, total_quantity, profit_margin]
})
display(kpis)"""))

cells.append(nbformat.v4.new_markdown_cell("## 5. Customer purchase volume and segment analysis"))
cells.append(nbformat.v4.new_code_cell("""customer_orders = (sales.groupby(["Customer ID", "Customer Name"])
                    .agg(Orders=("Order ID", "nunique"),
                         Sales=("Sales", "sum"),
                         Profit=("Profit", "sum"),
                         Units=("Quantity", "sum"))
                    .sort_values("Sales", ascending=False))

print("Top 10 customers by sales:")
display(customer_orders.head(10))

segment_summary = sales.groupby("Segment").agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Orders=("Order ID", "nunique"),
    Customers=("Customer ID", "nunique")
).sort_values("Sales", ascending=False)
display(segment_summary)

segment_summary["Sales"].plot(kind="bar", figsize=(8, 4), title="Sales by Customer Segment")
plt.ylabel("Sales ($)")
plt.xlabel("Customer Segment")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()"""))

cells.append(nbformat.v4.new_markdown_cell("## 6. Monthly sales seasonality"))
cells.append(nbformat.v4.new_code_cell("""monthly_sales = sales.groupby("Year-Month").agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Orders=("Order ID", "nunique")
).reset_index()

plt.figure(figsize=(14, 5))
sns.lineplot(data=monthly_sales, x="Year-Month", y="Sales", marker="o")
plt.title("Monthly Sales Trend")
plt.xlabel("Year-Month")
plt.ylabel("Sales ($)")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

# Average sales by calendar month across all years
month_order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
seasonality = sales.groupby("Month Name")["Sales"].sum().reindex(month_order)
seasonality.plot(kind="bar", figsize=(10, 4), title="Total Sales by Calendar Month")
plt.xlabel("Month")
plt.ylabel("Sales ($)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

display(monthly_sales.sort_values("Sales", ascending=False).head(10))"""))

cells.append(nbformat.v4.new_markdown_cell("## 7. Category-wise and sub-category revenue/profit"))
cells.append(nbformat.v4.new_code_cell("""category_summary = sales.groupby("Category").agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Quantity=("Quantity", "sum")
).sort_values("Sales", ascending=False)
category_summary["Profit Margin (%)"] = category_summary["Profit"] / category_summary["Sales"] * 100
display(category_summary)

category_summary["Sales"].plot(kind="bar", figsize=(8, 4), title="Revenue by Product Category")
plt.ylabel("Sales ($)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

subcategory_summary = sales.groupby(["Category", "Sub-Category"]).agg(
    Sales=("Sales", "sum"), Profit=("Profit", "sum")
).sort_values("Sales", ascending=False)
display(subcategory_summary)

plt.figure(figsize=(12, 6))
sns.barplot(data=subcategory_summary.reset_index(), x="Sales", y="Sub-Category", hue="Category", dodge=False)
plt.title("Sales by Sub-Category")
plt.xlabel("Sales ($)")
plt.ylabel("Sub-Category")
plt.tight_layout()
plt.show()"""))

cells.append(nbformat.v4.new_markdown_cell("## 8. Regional performance"))
cells.append(nbformat.v4.new_code_cell("""region_summary = sales.groupby("Region").agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Orders=("Order ID", "nunique")
).sort_values("Sales", ascending=False)
region_summary["Profit Margin (%)"] = region_summary["Profit"] / region_summary["Sales"] * 100
display(region_summary)

region_summary[["Sales", "Profit"]].plot(kind="bar", figsize=(9, 5), title="Sales and Profit by Region")
plt.ylabel("Amount ($)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()"""))

cells.append(nbformat.v4.new_markdown_cell("""## 9. Discount sensitivity and profitability

This section compares profit across discount levels. It is descriptive: discount levels may be associated with other factors (product, category, quantity, and customer mix), so the results do not prove that discount alone caused a profit change."""))
cells.append(nbformat.v4.new_code_cell("""discount_summary = sales.groupby("Discount").agg(
    Transactions=("Discount", "size"),
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Average_Profit=("Profit", "mean")
).reset_index()
discount_summary["Profit Margin (%)"] = discount_summary["Profit"] / discount_summary["Sales"] * 100
display(discount_summary)

plt.figure(figsize=(9, 5))
sns.scatterplot(data=sales, x="Discount", y="Profit", alpha=0.35)
plt.axhline(0, linestyle="--")
plt.title("Discount vs. Profit")
plt.xlabel("Discount")
plt.ylabel("Profit ($)")
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 5))
sns.boxplot(data=sales, x="Discount", y="Profit")
plt.axhline(0, linestyle="--")
plt.title("Profit Distribution by Discount Level")
plt.xlabel("Discount")
plt.ylabel("Profit ($)")
plt.tight_layout()
plt.show()"""))

cells.append(nbformat.v4.new_markdown_cell("## 10. Executive business findings"))
cells.append(nbformat.v4.new_code_cell("""# Generate evidence-backed findings from the cleaned data
best_category = category_summary["Sales"].idxmax()
best_category_sales = category_summary.loc[best_category, "Sales"]
best_profit_category = category_summary["Profit"].idxmax()

best_region = region_summary["Sales"].idxmax()
best_region_sales = region_summary.loc[best_region, "Sales"]

worst_discount = discount_summary.loc[discount_summary["Average_Profit"].idxmin()]
negative_discount_levels = discount_summary.loc[discount_summary["Profit"] < 0, "Discount"].tolist()

peak_month = monthly_sales.loc[monthly_sales["Sales"].idxmax()]
best_calendar_month = seasonality.idxmax()

print(f"1. The dataset contains {len(sales):,} valid sales transactions/lines after excluding {len(df)-len(sales):,} incomplete or non-sales rows.")
print(f"2. Total sales were ${total_sales:,.2f}, with total profit of ${total_profit:,.2f} and a profit margin of {profit_margin:.2f}%.")
print(f"3. {best_category} generated the highest category revenue (${best_category_sales:,.2f}); {best_profit_category} generated the highest total profit.")
print(f"4. {best_region} was the highest-revenue region (${best_region_sales:,.2f}).")
print(f"5. {best_calendar_month} had the highest cumulative sales across calendar months in the four-year period.")
print(f"6. The highest-sales individual month was {peak_month['Year-Month']} (${peak_month['Sales']:,.2f}).")
print(f"7. Discount level {worst_discount['Discount']:.0%} had the lowest average profit per line (${worst_discount['Average_Profit']:,.2f}).")
print(f"8. Discount levels with negative aggregate profit: {negative_discount_levels}. Review these discount bands before applying broad promotions.")"""))

cells.append(nbformat.v4.new_markdown_cell("""## 11. Conclusion and recommendations

- Prioritize profitable categories and sub-categories, not revenue alone.
- Review high-discount transactions where aggregate profit is negative.
- Plan inventory and marketing around the observed monthly sales peaks.
- Compare regional sales and profit margins when allocating sales resources.
- Use customer-level purchase history to identify high-value and repeat customers.

**Caution:** This is a historical, descriptive analysis. Validate findings against business context before making pricing or inventory decisions."""))

nb.cells = cells
nbformat.write(nb, out / "Superstore_Retail_Sales_EDA.ipynb")

readme = """# Superstore Retail Sales – EDA

## Files
- `Superstore_Retail_Sales_EDA.ipynb` – Jupyter Notebook containing data cleaning, KPI calculations, customer/segment analysis, monthly seasonality, category and regional performance, discount sensitivity, visualizations, and executive findings.
- `superstore_sales.csv` – Source dataset (keep it in the same folder as the notebook).

## Requirements
Install the required packages:

```bash
python -m pip install pandas numpy matplotlib seaborn jupyter
```

## Run the notebook
1. Place `Superstore_Retail_Sales_EDA.ipynb` and `superstore_sales.csv` in the same folder.
2. Open a terminal in that folder.
3. Run:

```bash
jupyter notebook
```

4. Open `Superstore_Retail_Sales_EDA.ipynb`.
5. Select **Kernel → Restart & Run All**.

## Data-cleaning note
The supplied CSV contains 10,800 rows. Only 9,994 rows have valid order dates and required numeric sales fields. The notebook parses dates and numeric columns, removes exact duplicates, and excludes incomplete/non-transaction rows from the sales analysis.

## Main analysis
- Data quality and missing-value profiling
- Customer purchase volume and segment performance
- Monthly sales trend and seasonality
- Category and sub-category sales/profit
- Regional performance
- Discount versus profit analysis
- Evidence-based executive findings
"""
(out / "README_Superstore_EDA.md").write_text(readme, encoding="utf-8")

print("Created:")
print(out / "Superstore_Retail_Sales_EDA.ipynb")
print(out / "README_Superstore_EDA.md")


FileNotFoundError: [Errno 2] No such file or directory: '\\mnt\\data\\Superstore_Retail_Sales_EDA.ipynb'